In [0]:
from functools import reduce
from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField, ArrayType, MapType,
                               StringType, LongType, DoubleType)
from pyspark.sql.window import Window

BRONZE = "/Volumes/workspace/bronze/raw_files/restaurants"
TABELA = "workspace.silver.restaurants"
REJEITADOS = "workspace.silver.restaurants_rejected"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABELA} (
  restaurant_id STRING,
  osm_type STRING,
  name STRING,
  amenity STRING,
  cuisine_raw STRING,
  cuisine_list ARRAY<STRING>,
  categoria_comida STRING,
  categoria_fonte STRING,
  lat DOUBLE,
  lon DOUBLE,
  opening_hours STRING,
  has_opening_hours BOOLEAN,
  bairro STRING,
  ingestion_date DATE,
  source_file STRING,
  updated_at TIMESTAMP
)
""")

In [0]:
schema = StructType([StructField("elements", ArrayType(StructType([
    StructField("type", StringType()),
    StructField("id", LongType()),
    StructField("lat", DoubleType()),
    StructField("lon", DoubleType()),
    StructField("center", StructType([
        StructField("lat", DoubleType()),
        StructField("lon", DoubleType()),
    ])),
    StructField("tags", MapType(StringType(), StringType())),
])))])

raw = (spark.read.schema(schema).option("multiLine", True).json(BRONZE)
       .withColumn("source_file", F.col("_metadata.file_path")))

df = (raw.select("source_file", F.explode("elements").alias("e"))
      .select(
          F.concat_ws("/", F.col("e.type"), F.col("e.id").cast("string")).alias("restaurant_id"),
          F.col("e.type").alias("osm_type"),
          F.trim(F.col("e.tags")["name"]).alias("name"),
          F.col("e.tags")["amenity"].alias("amenity"),
          F.col("e.tags")["cuisine"].alias("cuisine_raw"),
          F.coalesce("e.lat", "e.center.lat").alias("lat"),
          F.coalesce("e.lon", "e.center.lon").alias("lon"),
          F.col("e.tags")["opening_hours"].alias("opening_hours"),
          F.col("e.tags")["addr:suburb"].alias("bairro"),
          F.to_date(F.regexp_extract("source_file", r"ingestion_date=(\d{4}-\d{2}-\d{2})", 1)).alias("ingestion_date"),
          "source_file",
      ))

# Se o mesmo estabelecimento veio em mais de um dia, fica com o mais recente
w = (Window.partitionBy("restaurant_id")
     .orderBy(F.col("ingestion_date").desc(), F.col("source_file").desc()))
df = df.withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn")

print("Estabelecimentos únicos:", df.count())

In [0]:
CUISINE_RULES = [
    ("sopas_caldos",     ["soup", "caldo", "broth"]),
    ("sorvete_acai",     ["ice cream", "frozen yogurt", "acai", "gelato", "milkshake"]),
    ("massas_pizza",     ["pizza", "italian", "pasta"]),
    ("hamburguer",       ["burger", "hamburger"]),
    ("asiatica",         ["sushi", "japan", "japon", "asian", "chinese", "korean", "thai", "ramen",
                          "vietnamese", "poke", "hawaiian", "indian", "udon", "yakisoba", "oriental"]),
    ("churrasco_carnes", ["steak", "barbecue", "churrasc","grill", "bbq", "gaucho"]),
    ("brasileira",       ["regional", "brazilian", "brasileira", "feijoada", "mineira", "nordestina", "marmita"]),
    ("cafe_padaria",     ["coffee", "cafe", "bakery", "breakfast", "tea", "padaria", "bread"]),
    ("doces_sobremesas", ["dessert", "cake", "pastry", "donut", "chocolate", "waffle", "doceria","crepe"]),
    ("lanches",          ["sandwich", "chicken", "hot dog", "kebab", "empanada", "snack", "fast food",
                          "wings", "pastel", "salgado", "american", "pretzel", "fried food", "shawarma"]),
    ("saudavel",         ["salad", "vegetarian", "vegan", "healthy", "juice", "suco"]),
    ("frutos_do_mar",    ["seafood", "fish"]),
    ("internacional",    ["mexican", "tex-mex", "tex mex", "french", "german", "spanish", "portuguese",
                          "peruvian", "colombian", "arab", "middle eastern", "syrian", "lebanese",
                          "mediterranean", "argentinian", "greek", "international"]),
]
NAME_RULES = [
    ("sopas_caldos",     ["sopa", "caldo"]),
    ("sorvete_acai",     ["acai", "sorvete", "gelato", "gelateria", "picole", "sundae"]),
    ("massas_pizza",     ["pizza", "pasta", "massa", "cantina", "trattoria", "italian", "macarrao", "trattoria","lasanha"]),
    ("hamburguer",       ["hamburg", "burger", "burguer", "smash"]),
    ("asiatica",         ["sushi", "temaki", "japon", "yakisoba", "lamen", "ramen", "oriental", "chines", "thai", "coreano", "poke"]),
    ("churrasco_carnes", ["churrasc", "steak", "grill", "parrilla", "espeto", "picanha", "bbq", "assador", "carnes", "rodizio", "beef"]),
    ("brasileira",       ["feijoada", "mineir", "nordestin", "baiano", "caseira", "marmita", "por quilo", "self service"]),
    ("cafe_padaria",     ["cafe", "padaria", "panificadora", "coffee", "boulang", r"tea\b"]),
    ("doces_sobremesas", ["confeitaria", "doceria", "brigadeiro", "chocolate", "bolo", "cupcake", "donut",
                          "crepe", "waffle", "doces", r"doce\b", "sweet"]),
    ("lanches",          ["lanche", "lanchon", "pastel", "coxinha", "esfiha", "salgado", "hot dog", "frango", "kebab", "sanduiche"]),
    ("saudavel",         ["salada", "natural", "vegano", "vegetariano", r"fit\b", "suco", "saudavel", "fruta"]),
    ("frutos_do_mar",    ["peixe", "frutos do mar", "camarao", "seafood", "mexilh", "bacalhau", "sereia" , "marisco"]),
    ("internacional",    ["libanes", "arabe", "mexican", "frances", "peruan", "portugues", "argentin"]),
]
AMENITY_MAP = {
    "ice_cream": "sorvete_acai",
    "cafe": "cafe_padaria",
    "fast_food": "lanches",
    "restaurant": "geral",
}
def regras_para_expr(coluna, regras):
    """Primeira regra que casar vence. Casa o começo da palavra (\\b)."""
    exprs = [(F.col(coluna).rlike(r"\b(" + "|".join(p) + ")"), F.lit(cat)) for cat, p in regras]
    return reduce(lambda acc, x: acc.when(x[0], x[1]), exprs[1:], F.when(*exprs[0]))

# Textos normalizados (minúsculo e sem acento) - agora o cuisine também
acentos_de = "áàâãäéèêëíìîïóòôõöúùûüç"
acentos_para = "aaaaaeeeeiiiiooooouuuuc"
df = (df
      .withColumn("_cuisine_txt", F.regexp_replace(
          F.translate(F.lower(F.coalesce("cuisine_raw", F.lit(""))), acentos_de, acentos_para), "[;_]", " "))
      .withColumn("_name_txt", F.translate(F.lower(F.coalesce("name", F.lit(""))), acentos_de, acentos_para)))
df = (df
      .withColumn("_cat_cuisine", regras_para_expr("_cuisine_txt", CUISINE_RULES))
      .withColumn("_cat_nome", regras_para_expr("_name_txt", NAME_RULES))
      .withColumn("_cat_amenity", F.create_map(*[x for k, v in AMENITY_MAP.items() for x in (F.lit(k), F.lit(v))])[F.col("amenity")]))
df = (df
      .withColumn("categoria_comida", F.coalesce("_cat_cuisine", "_cat_nome", "_cat_amenity", F.lit("geral")))
      .withColumn("categoria_fonte",
                  F.when(F.col("_cat_cuisine").isNotNull(), "cuisine")
                   .when(F.col("_cat_nome").isNotNull(), "nome")
                   .when(F.col("_cat_amenity").isNotNull(), "amenity")
                   .otherwise("default"))
      .withColumn("cuisine_list",
                  F.array_remove(F.split(F.lower(F.coalesce("cuisine_raw", F.lit(""))), r"\s*;\s*"), ""))
      .withColumn("has_opening_hours", F.col("opening_hours").isNotNull()))
display(df.groupBy("categoria_comida", "categoria_fonte").count().orderBy(F.desc("count")))

In [0]:
motivo = (F.when(F.col("restaurant_id").isNull(), "id_ausente")
           .when(F.col("name").isNull() | (F.col("name") == ""), "sem_nome")
           .when(F.col("lat").isNull() | F.col("lon").isNull(), "sem_coordenada")
           .when(~(F.col("lat").between(-24.1, -23.0) & F.col("lon").between(-47.0, -46.0)), "fora_de_sp")
           .otherwise(None))

df = df.withColumn("motivo_rejeicao", motivo)

colunas = ["restaurant_id", "osm_type", "name", "amenity", "cuisine_raw", "cuisine_list",
           "categoria_comida", "categoria_fonte", "lat", "lon", "opening_hours",
           "has_opening_hours", "bairro", "ingestion_date", "source_file"]

validos = df.filter("motivo_rejeicao IS NULL").select(*colunas).withColumn("updated_at", F.current_timestamp())
rejeitados = df.filter("motivo_rejeicao IS NOT NULL").select(*colunas, "motivo_rejeicao") \
               .withColumn("rejected_at", F.current_timestamp())

print("Válidos:", validos.count(), "| Rejeitados:", rejeitados.count())
display(rejeitados.groupBy("motivo_rejeicao").count())

(rejeitados.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(REJEITADOS))

In [0]:
validos.createOrReplaceTempView("restaurants_novo")

spark.sql(f"""
MERGE INTO {TABELA} AS t
USING restaurants_novo AS s
ON t.restaurant_id = s.restaurant_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *
""")

In [0]:
display(spark.sql(f"""
SELECT categoria_comida,
       COUNT(*) AS total,
       SUM(CASE WHEN categoria_fonte = 'cuisine' THEN 1 ELSE 0 END) AS via_cuisine,
       SUM(CASE WHEN categoria_fonte = 'nome' THEN 1 ELSE 0 END) AS via_nome,
       SUM(CASE WHEN categoria_fonte IN ('amenity','default') THEN 1 ELSE 0 END) AS via_amenity
FROM {TABELA}
GROUP BY categoria_comida
ORDER BY total DESC
"""))

In [0]:
silver = spark.table(TABELA)   # workspace.silver.restaurants

# 1) Total de linhas e duplicados (os dois números abaixo têm que ser iguais)
print("Total de linhas:      ", silver.count())
print("restaurant_id únicos: ", silver.select("restaurant_id").distinct().count())

# 2) Distribuição por categoria e por fonte da classificação, com percentual
total = silver.count()
display(
    silver.groupBy("categoria_comida", "categoria_fonte").count()
          .withColumn("pct", F.round(F.col("count") / total * 100, 1))
          .orderBy(F.desc("count"))
)

# 3) Nulos nas colunas principais
display(
    silver.select([F.sum(F.col(c).isNull().cast("int")).alias(c)
                   for c in ["restaurant_id", "name", "categoria_comida", "categoria_fonte"]])
)

# 4) Amostra final pra olho humano
display(silver.select("name", "amenity", "cuisine_raw", "categoria_comida", "categoria_fonte")
              .orderBy(F.rand()).limit(20))